# Analyse rhétorique collaborative par agents IA — exécuteur principal


[Retour au README](../README.md) | [Précédent : Orchestration](./Argumentation-07-Orchestration-Python.ipynb)

**Objectif:** Ce notebook orchestre et exécute une analyse rhétorique multi-agents sur un texte donné. Il sert de point d'entrée principal pour lancer le processus.

**Structure Modulaire:**
1.  `Argumentation-08c-UI-Configuration-Python.ipynb` : Gère l'interface utilisateur pour sélectionner/préparer le texte à analyser (incluant sources prédéfinies, URL, fichier, texte direct, et extraction) et charge/sauvegarde la configuration des sources.
2.  `Argumentation-00-Setup-Tweety-Python.ipynb`: Configuration initiale (dépendances, LLM, JVM), définition de l'état partagé (`RhetoricalAnalysisState`) et du gestionnaire d'état (`StateManagerPlugin`).
3.  `Argument_Analysis_Agentic-1-informal_agent.ipynb`: Définition de l'`InformalAnalysisAgent`.
4.  `Argument_Analysis_Agentic-2-pl_agent.ipynb`: Définition du `PropositionalLogicAgent`.
5.  `Argumentation-07-Orchestration-Python.ipynb`: Définition des stratégies d'orchestration et de la fonction principale `run_analysis_conversation`.

**Prérequis:**
* Un fichier `.env` à la racine contenant les clés API, configurations LLM, et la clé de chiffrement `TEXT_CONFIG_KEY`.
* Un environnement Java Development Kit (JDK >= 11) correctement installé et configuré (`JAVA_HOME`).
* Les dépendances Python installées (`ipywidgets`, `requests`, `jupyter-ui-poll`, `python-dotenv`, `semantic-kernel`, `pandas`, `jpype1`, `cryptography`).
* Les JARs Tweety placés dans le dossier `libs/`.
* Le fichier `extract_sources.json.gz.enc` (s'il existe déjà) contenant les définitions des sources.

## Présentation générale du système

### Objectifs pédagogiques

Ce notebook illustre une architecture d'**agents collaboratifs multi-modaux** pour l'analyse rhétorique automatisée. Les étudiants découvriront:

1. **Orchestration d'agents IA** (Semantic Kernel)
2. **Analyse hybride**: LLM (compréhension naturelle) + Logique formelle (preuve)
3. **État partagé** pour communication inter-agents
4. **Validation structurée** avec métriques de confiance

### Architecture du système

Le système combine deux paradigmes d'IA complémentaires:

| Composant | Technologie | Rôle |
|-----------|-------------|------|
| **InformalAnalysisAgent** | LLM (GPT-4/Claude) | Comprend langage naturel, identifie arguments/sophismes |
| **PropositionalLogicAgent** | Tweety (bibliothèque Java) | Formalise en logique PL, prouve cohérence/implications |
| **StateManager** | Dataclass Python | Mémoire partagée entre agents |
| **Orchestration** | Semantic Kernel | Coordination des appels d'agents |

### Workflow en 3 phases

```
Phase 1: ANALYSE INFORMELLE
├─ Détection arguments (prémisses, conclusions)
├─ Identification sophismes (taxonomie hiérarchique)
└─ Stockage état (JSON structuré)

Phase 2: FORMALISATION LOGIQUE
├─ Traduction propositions en PL
├─ Création Belief Sets (Tweety)
├─ Requêtes logiques (cohérence, implication)
└─ Stockage preuves formelles

Phase 3: VALIDATION & SYNTHÈSE
├─ Cross-validation (6 critères)
├─ Calcul score de confiance
├─ Génération conclusion synthétique
└─ Export rapport JSON
```

> **Cas d'usage réel**: Ce système peut analyser des débats politiques, détecter la manipulation rhétorique, ou servir d'outil pédagogique pour l'enseignement de la pensée critique.

## 1. Chargement de l'Environnement

Chargement des variables depuis le fichier `.env` (clés API, clé de chiffrement, etc.).

### Rôle du fichier .env

Le fichier `.env` centralise toutes les configurations sensibles et paramètres du système:

| Variable | Fonction |
|----------|----------|
| `OPENAI_API_KEY` | Authentification API OpenAI pour les LLMs |
| `ANTHROPIC_API_KEY` | Authentification API Anthropic (optionnel) |
| `TEXT_CONFIG_KEY` | Clé de chiffrement pour la config sécurisée |
| `BATCH_MODE` | Active le mode non-interactif pour tests automatisés |
| `BATCH_TEXT` | Texte personnalisé pour mode batch (optionnel) |

> **Note de sécurité**: Le fichier `.env` ne doit jamais être versionné avec Git. Utilisez `.env.example` comme template.

In [1]:
# Charger les variables d'environnement (résolution de chemin robuste)
# CRITIQUE 1 : find_dotenv() dépend du CWD courant,
# qui peut differer de Argument_Analysis/ (Papermill, MCP Jupyter, CI changent
# le working directory). On localise Argument_Analysis/ par walk-up depuis le
# CWD, puis on s'y positionne (os.chdir) pour que tous les %run relatifs, le
# dossier libs/ (JARs Tweety), data/ et extract_sources.json.gz.enc résolvent
# correctement quel que soit le point de départ.
import os
from pathlib import Path
from dotenv import load_dotenv

def _find_argument_analysis_dir(start: Path, max_up: int = 6) -> Path:
    """Localise Argument_Analysis/ via son marqueur, en remontant depuis start."""
    marker = "Argumentation-08b-Executor-Python.ipynb"
    candidate = start.resolve()
    for _ in range(max_up):
        # Cas direct : on est déjà dans Argument_Analysis/
        if (candidate / marker).exists():
            return candidate
        # Cas nested : Argument_Analysis/ est un sous-dossier
        if (candidate / "Argument_Analysis" / marker).exists():
            return candidate / "Argument_Analysis"
        if len(candidate.parts) > 1:
            candidate = candidate.parent
        else:
            break
    raise RuntimeError(
        f"Dossier Argument_Analysis/ introuvable (marqueur={marker}) "
        f"en remontant depuis {start}"
    )

AA_DIR = _find_argument_analysis_dir(Path.cwd())
os.chdir(str(AA_DIR))  # %run relatifs + libs/ + data/ résolvent depuis ici
loaded_env = load_dotenv(AA_DIR / ".env", override=True)

print(f"CWD -> {os.getcwd()}")
print(f".env chargé: {loaded_env}")  # True si le .env a été trouvé et chargé

CWD -> D:\dev\CoursIA-arg-arc1\MyIA.AI.Notebooks\SymbolicAI\Argument_Analysis
.env chargé: True


### Interprétation du résultat

**Sortie attendue**: `.env chargé: True`

Si vous obtenez `False`, cela signifie:
- Le fichier `.env` n'existe pas dans le répertoire courant ou ses parents
- Les variables d'environnement doivent être définies manuellement

**Actions correctives**:
1. Copier `.env.example` vers `.env`
2. Remplir les valeurs manquantes (clés API)
3. Relancer cette cellule

### Vérification des dépendances

Avant de continuer, assurez-vous que les dépendances suivantes sont satisfaites:

**Dépendances Python**:
```bash
pip install python-dotenv semantic-kernel pandas jpype1 cryptography
pip install ipywidgets requests jupyter-ui-poll  # Pour mode interactif
```

**Dépendances système**:
- **JDK 11+** (pour Tweety via JPype): `JAVA_HOME` doit être configuré
- **Bibliothèque Tweety**: JARs dans `libs/` (téléchargement automatique via `Tweety-01-Setup-Python.ipynb`)

**Fichiers requis**:

| Fichier | Rôle | Création |
|---------|------|----------|
| `.env` | Variables d'environnement (clés API) | Copier depuis `.env.example` |
| `libs/*.jar` | Bibliothèque Tweety Java | Téléchargement auto |
| `data/logical_fallacies_taxonomy.json` | Taxonomie sophismes | Fourni |

> **Astuce**: Si JPype échoue à s'initialiser, le système basculera en mode "fallback" (sans logique formelle). L'analyse informelle fonctionnera toujours.

## 2. Chargement de l'Interface Utilisateur

Exécution du notebook `UI_Configuration.ipynb` pour définir la fonction `configure_analysis_task()`. C'est ce notebook qui contient désormais toute la logique de l'interface graphique, du cache fichier et de la gestion de la configuration chiffrée.

In [2]:
# Mode batch contrôlé par variable d'environnement
# Si BATCH_MODE=true dans .env, on skip l'UI interactive (widgets bloquants)
import os
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() in ("true", "1", "yes")

if BATCH_MODE:
    print("Mode BATCH detecte (BATCH_MODE=true dans .env)")
    print("   -> Skip du chargement UI_configuration.ipynb (widgets non compatibles)")
    print("   -> Le texte sera fourni directement dans la cellule suivante")
else:
    # Executer le notebook UI pour definir la fonction configure_analysis_task
    # Resolution du chemin absolu (Papermill change le working directory)
    import os
    _notebook_dir = os.path.dirname(os.path.abspath("Argumentation-08b-Executor-Python.ipynb")) if os.path.exists("Argumentation-08b-Executor-Python.ipynb") else os.getcwd()
    _ui_nb_path = os.path.join(_notebook_dir, "Argumentation-08c-UI-Configuration-Python.ipynb")
    if os.path.exists(_ui_nb_path):
        print(f"Execution de {_ui_nb_path}...")
        get_ipython().run_line_magic("run", _ui_nb_path)
        print("Execution de Argumentation-08c-UI-Configuration-Python.ipynb terminee.")
    else:
        print(f"ATTENTION: {_ui_nb_path} non trouve. Passage en mode batch.")
        BATCH_MODE = True

    # Vérification que la fonction est bien définie après l'exécution
    if 'configure_analysis_task' not in locals():
        print("ERREUR CRITIQUE : La fonction configure_analysis_task n'a pas été définie par UI_Configuration.ipynb !")
    else:
        print("Fonction configure_analysis_task trouvée.")


Mode BATCH detecte (BATCH_MODE=true dans .env)
   -> Skip du chargement UI_configuration.ipynb (widgets non compatibles)
   -> Le texte sera fourni directement dans la cellule suivante


### Modes d'exécution: Batch vs Interactif

Le notebook supporte deux modes d'exécution distincts:

| Mode | Activation | Usage | Interface |
|------|------------|-------|-----------|
| **Interactif** | `BATCH_MODE=false` (défaut) | Démonstration, exploration | Widgets Jupyter (ipywidgets) |
| **Batch** | `BATCH_MODE=true` dans `.env` | Tests automatisés, CI/CD, Papermill | Texte prédéfini sans interaction |

**Pourquoi le mode batch?**

Les widgets Jupyter (`ipywidgets`) utilisent des boucles de polling bloquantes incompatibles avec:
- L'exécution Papermill
- Les MCP (Model Context Protocol) Jupyter
- Les pipelines CI/CD automatisés

Le mode batch permet de contourner ces limitations en sautant le chargement de `UI_configuration.ipynb` et en utilisant directement un texte source.

> **Note technique**: En mode batch, le texte est fourni via `BATCH_TEXT` (variable d'environnement) ou via le texte d'exemple enrichi défini dans la cellule suivante.

### Exercice 1 : Gestion de configuration avec fallback

L'Executor lit des variables d'environnement pour configurer l'analyse. Implementez un mécanisme de fallback robuste.

**Objectif** : Ecrire `get_config_with_fallback`.

**Indices** :
- **Étape 1** : Utiliser `os.getenv(key)` pour lire la variable
- **Étape 2** : Verifier si le résultat est `None` ou une chaîne vide
- **Étape 3** : Retourner la valeur par defaut si la variable est absente
- **Étape 4** : Tester avec une variable qui n'existe pas

In [3]:
# Exercice 1 : Gestion de configuration avec fallback
# TODO etudiant: ecrivez une fonction qui lit une variable d'environnement
# et retourne une valeur par defaut si absente.
# Etape 1: Utiliser os.getenv pour lire la variable
# Etape 2: Verifier si la valeur est None ou vide
# Etape 3: Retourner le fallback si necessaire
# Etape 4: Tester avec une variable inexistante
import os
def get_config_with_fallback(key, default="Texte par defaut"):
    return default  # TODO etudiant

text = get_config_with_fallback("BATCH_TEXT_EXERCICE", "Exemple de texte")
print(f"Texte choisi: {text}")
print("Exercice a completer")


Texte choisi: Exemple de texte
Exercice a completer


## 3. Configuration de la Tâche et Récupération du Texte

### Sélection du texte à analyser

Cette étape configure le texte source selon le mode d'exécution:

**Mode Interactif**:
- Interface graphique avec widgets Jupyter
- Choix entre sources prédéfinies, URL, fichier, ou texte direct
- Extraction de contenu (HTML, PDF, etc.)
- Configuration chiffrée sauvegardée (`extract_sources.json.gz.enc`)

**Mode Batch**:
- Texte fourni via variable d'environnement `BATCH_TEXT`
- Ou texte d'exemple prédéfini (débat sur transition énergétique)
- Pas d'interaction utilisateur requise

> **Attention**: En mode interactif, la cellule suivante attendra que vous cliquiez sur "Lancer l'Analyse" dans l'interface. Ne passez pas à la suite avant d'avoir sélectionné un texte.

In [4]:
# Configuration du texte pour l'analyse
# En mode BATCH : utilise BATCH_TEXT (variable d'env) ou le texte d'exemple
# En mode interactif : appelle configure_analysis_task() (si disponible)

texte_pour_analyse = None

# Texte d'exemple enrichi pour le mode batch - contient des arguments complexes et des sophismes
TEXTE_EXEMPLE_BATCH = """Le debat sur la transition energetique en France revele des enjeux majeurs pour notre avenir.

ARGUMENTS EN FAVEUR DES ENERGIES RENOUVELABLES:
Les energies renouvelables sont essentielles pour lutter contre le changement climatique. La France s'est engagee a reduire ses emissions de CO2 de 40% d'ici 2030, ce qui necessite un investissement massif dans le solaire et l'eolien. D'ailleurs, l'Allemagne a deja prouve que c'etait possible en produisant 46% de son electricite a partir de sources renouvelables en 2023.

De plus, les energies renouvelables creent des emplois locaux non delocalisables. Selon l'ADEME, le secteur emploie deja plus de 150 000 personnes en France et pourrait en creer 500 000 supplementaires d'ici 2050. Quiconque s'oppose a cette transition est donc contre la creation d'emplois.

ARGUMENTS CONTRE LE DEVELOPPEMENT MASSIF DES RENOUVELABLES:
Cependant, les ecologistes veulent nous ramener a l'age de pierre en supprimant le nucleaire, qui fournit pourtant 70% de notre electricite. Sans le nucleaire, nous serions obliges de choisir entre des coupures d'electricite massives ou une dependance totale au gaz russe.

Le cout de la transition est exorbitant. Tout le monde sait que les eoliennes sont inefficaces car elles ne tournent que 25% du temps. Mon voisin Jean-Pierre, qui est ingenieur a EDF, m'a dit que les renouvelables ne pourront jamais remplacer le nucleaire. C'est un expert, donc il a forcemment raison.

De plus, les panneaux solaires sont fabriques en Chine avec du charbon, ce qui annule completement leur benefice ecologique. Donc investir dans le solaire revient a financer la pollution chinoise.

CONCLUSION:
Il faut arreter de debattre et agir. Soit on accepte une transition energetique complete vers 100% de renouvelables d'ici 2040, soit on condamne nos enfants a vivre sur une planete inhabitable. Des milliers de scientifiques ont signe une petition pour le climat, ce qui prouve definitivement que nous avons raison de promouvoir les energies vertes. Les climatosceptiques sont finances par les lobbies petroliers et ne meritent pas qu'on ecoute leurs arguments.

En conclusion, bien que certains arguments des deux cotes aient du merite, la question necessite une analyse nuancee des compromis entre independance energetique, impact environnemental, cout economique et faisabilite technique."""

if BATCH_MODE:
    # === MODE BATCH ===
    print("Mode BATCH - Configuration automatique du texte")
    
    # Priorite 1: Variable d'environnement BATCH_TEXT
    batch_text_env = os.getenv("BATCH_TEXT", "")
    if batch_text_env:
        texte_pour_analyse = batch_text_env
        print(f"Texte charge depuis BATCH_TEXT ({len(texte_pour_analyse)} caracteres)")
    else:
        # Priorite 2: Texte d'exemple par defaut
        texte_pour_analyse = TEXTE_EXEMPLE_BATCH
        print(f"Texte d'exemple enrichi utilise ({len(texte_pour_analyse)} caracteres)")
    
    print(f"\nExtrait du texte:\n{texte_pour_analyse[:200]}...")

else:
    # === MODE INTERACTIF ===
    print("Mode INTERACTIF - Lancement de l'interface de configuration")
    
    if 'configure_analysis_task' in locals():
        try:
            texte_pour_analyse = configure_analysis_task()
            print(f"Texte recupere via l'interface ({len(texte_pour_analyse) if texte_pour_analyse else 0} caracteres)")
        except Exception as e_ui:
            print(f"Erreur lors de la configuration UI : {e_ui}")
            import traceback
            traceback.print_exc()
    else:
        print("Fonction configure_analysis_task non disponible - verifiez le chargement de UI_configuration.ipynb")

# Verification finale
if not texte_pour_analyse:
    print("\nAucun texte prepare. L'analyse ne peut pas continuer.")
else:
    print(f"\nTexte pret pour l'analyse (longueur: {len(texte_pour_analyse)}). Passage au chargement des agents.")


Mode BATCH - Configuration automatique du texte
Texte d'exemple enrichi utilise (2348 caracteres)

Extrait du texte:
Le debat sur la transition energetique en France revele des enjeux majeurs pour notre avenir.

ARGUMENTS EN FAVEUR DES ENERGIES RENOUVELABLES:
Les energies renouvelables sont essentielles pour lutter ...

Texte pret pour l'analyse (longueur: 2348). Passage au chargement des agents.


### Interprétation de la configuration du texte

**Résultat attendu (Mode Batch)**:
```
Mode BATCH - Configuration automatique du texte
Texte charge depuis BATCH_TEXT (XXX caractères)
ou
Texte d'exemple enrichi utilise (XXX caractères)
```

**Résultat attendu (Mode Interactif)**:
```
Mode INTERACTIF - Lancement de l'interface de configuration
Texte recupere via l'interface (XXX caractères)
```

**Structure du texte d'exemple**:

Le texte d'exemple enrichi contient intentionnellement plusieurs sophismes pour démonstration:

| Sophisme | Exemple dans le texte |
|----------|----------------------|
| Faux dilemme | "Soit 100% renouvelables, soit planète inhabitable" |
| Appel à l'autorité | "Mon voisin ingénieur EDF dit que..." |
| Ad hominem | "Climatosceptiques financés par lobbies pétroliers" |
| Généralisation hâtive | "Tout le monde sait que les éoliennes..." |
| Pente glissante | "Les écologistes veulent nous ramener à l'âge de pierre" |

> **Conseil pédagogique**: Le texte d'exemple est conçu pour tester la robustesse de l'analyse rhétorique multi-agents. Utilisez-le pour comprendre comment les agents détectent les arguments fallacieux.

## 4. Chargement des Définitions des Agents et de l'Orchestration

### Architecture modulaire par notebooks

Le système est structuré en **4 notebooks enfants** chargés dynamiquement via `%run`:

**Avantages de cette architecture**:
1. **Séparation des responsabilités**: Chaque agent dans son propre fichier
2. **Réutilisabilité**: Les agents peuvent être chargés individuellement
3. **Maintenabilité**: Modification d'un agent sans toucher aux autres
4. **Testabilité**: Tests unitaires par agent

**Ordre de chargement critique**:

```
0-init.ipynb           (État + Config LLM + JVM)
    ↓
1-informal_agent.ipynb (Dépend de RhetoricalAnalysisState)
    ↓
2-pl_agent.ipynb       (Dépend de RhetoricalAnalysisState + JVM)
    ↓
3-orchestration.ipynb  (Dépend de tous les agents)
```

> **Note technique**: `%run` exécute le notebook dans le namespace courant. Toutes les variables définies dans les notebooks enfants (classes, fonctions) deviennent disponibles ici.

In [5]:
# Executer les notebooks enfants pour charger les definitions
# Seulement si un texte a ete prepare avec succes
#
# CRITIQUE 2 : les notebooks enfants (notamment
# 0-init_agent.ipynb) font leur propre os.chdir(TWEETY_DIR) pour localiser les
# JARs. Ce chdir "fuit" hors du %run et corrompt le CWD pour les %run suivants
# (relatifs a Argument_Analysis/) : 1-informal_agent.ipynb etait cherche dans
# SymbolicAI/Tweety/ et introuvable. On re-ancre le CWD sur AA_DIR avant CHAQUE
# %run pour garantir que les chemins relatifs resolvent quel que soit l'endroit
# ou l'enfant s'est positionne.
#
# CRITIQUE 3 : la chaine %run doit etre COHERENTE.
# Les notebooks "rungs" recents (0-init.ipynb, 3-orchestration.ipynb) ne
# construisent NI le service OpenAI NI run_analysis_conversation -- or la
# cellule 18 appelle run_analysis_conversation() et les agents charges sont les
# versions _agent (InformalAnalysisAgent, PropositionalLogicAgent). On charge
# donc la chaine LEGACY "_agent" auto-coherente : 0-init_agent (JVM + service
# OpenAI + kernel) -> 1-informal_agent (InformalAnalysisAgent) ->
# 2-pl_agent (PropositionalLogicAgent) -> 3-orchestration_agent
# (run_analysis_conversation). Cette chaine definit TOUT ce dont la cellule 18
# a besoin.
if 'texte_pour_analyse' in locals() and texte_pour_analyse:
    print("Chargement des definitions des agents et de l'orchestration...")
    try:
        os.chdir(str(AA_DIR))
        %run ./Argument_Analysis_Agentic-0-init_agent.ipynb
        os.chdir(str(AA_DIR))
        %run ./Argument_Analysis_Agentic-1-informal_agent.ipynb
        os.chdir(str(AA_DIR))
        %run ./Argument_Analysis_Agentic-2-pl_agent.ipynb
        os.chdir(str(AA_DIR))
        %run ./Argument_Analysis_Agentic-3-orchestration_agent.ipynb
        os.chdir(str(AA_DIR))
        print("[OK] Definitions chargees.")
        if 'run_analysis_conversation' not in locals():
            print("[ERREUR CRITIQUE] La fonction run_analysis_conversation n'a pas ete definie apres l'execution des notebooks agents !")
    except Exception as e_run:
        print("[ERREUR] Une erreur est survenue lors de l'execution des notebooks enfants : " + str(e_run))
        import traceback
        traceback.print_exc()
        # Empecher la suite si le chargement echoue
        texte_pour_analyse = None
        os.chdir(str(AA_DIR))
else:
    print("Skipping agent definition loading because no text was prepared.")


Chargement des definitions des agents et de l'orchestration...


--- Verification compatibilite semantic-kernel/openai ---


01:20:08 [INFO] [Orchestration.Setup] --- Vérification des dépendances ---


01:20:08 [INFO] [Orchestration.Setup] ✔️ Dépendance 'jpype' trouvée.


01:20:08 [INFO] [Orchestration.Setup] ✔️ Dépendance 'semantic_kernel' trouvée.


...

01:20:09 [INFO] [Orchestration.AgentPL.Instructions] Instructions Systeme PL_AGENT_INSTRUCTIONS (V16 - INTERDIT add_analysis_task) definies.


01:20:09 [INFO] [root] Classe SimpleTerminationStrategy définie.


01:20:09 [INFO] [root] Classe DelegatingSelectionStrategy (V2) définie.


01:20:10 [INFO] [Orchestration.Run] Verifications prealables OK.


Fonction setup_pm_kernel definie : configuration kernel ProjectManager avec plugins et fonctions semantiques
Classe InformalAnalysisPlugin definie avec ses methodes
Nombre de sous-categories: 0
Exercice a completer
Fonction setup_informal_kernel et prompt semantique prompt_identify_args_v7 definis
Analyse: {}
Exercice a completer
Resume du sophisme: 
Exercice a completer
Instructions systeme Informal configurees
Exercice a completer
Classe PropositionalLogicPlugin definie avec ses methodes
Formule PL: 
Exercice a completer
Fonction setup_pl_agent et prompts semantiques PL definis (3 prompts V10 + setup_pl_kernel V12)
Belief set: 
Exercice a completer
Instructions systeme PL configurees
Contradiction detectee: False
Exercice a completer
Exercice a completer
Arret sur mot-cle: False
Exercice a completer
Tour 0: None
Tour 1: None
Tour 2: None
Tour 3: None
Exercice a completer
{'fallacies': 0, 'conclusions': []}
Exercice a completer
[OK] Definitions chargees.


### Architecture modulaire chargée

**Notebooks chargés et leurs responsabilités**:

| Notebook | Composants définis | Rôle |
|----------|-------------------|------|
| `Agentic-0-init.ipynb` | `RhetoricalAnalysisState`, `StateManagerPlugin`, kernel SK | État partagé, configuration LLM, JVM Tweety |
| `Agentic-1-informal_agent.ipynb` | `InformalAnalysisAgent` | Analyse arguments et sophismes (taxonomie) |
| `Agentic-2-pl_agent.ipynb` | `PropositionalLogicAgent`, `PropositionalLogicPlugin` | Formalisation en logique propositionnelle |
| `Agentic-3-orchestration.ipynb` | `run_analysis_conversation()` | Orchestration collaborative des agents |

**Architecture technique**:

```
┌─────────────────────────────────────────────────┐
│  Semantic Kernel (orchestration)                │
│  ┌──────────────┐  ┌────────────────────────┐  │
│  │ Informal     │  │ Propositional Logic    │  │
│  │ Analysis     │─▶│ Agent (Tweety/JPype)   │  │
│  │ Agent        │  │                        │  │
│  └──────────────┘  └────────────────────────┘  │
│         │                     │                 │
│         └─────────┬───────────┘                 │
│                   ▼                             │
│        ┌────────────────────┐                   │
│        │ StateManager       │                   │
│        │ Plugin             │                   │
│        │ (état partagé)     │                   │
│        └────────────────────┘                   │
└─────────────────────────────────────────────────┘
```

> **Note d'implémentation**: L'état partagé (`RhetoricalAnalysisState`) permet aux agents de communiquer leurs résultats via un contexte commun plutôt que par messages directs.

## 5. Exécution de l'Analyse Collaborative

### Orchestration asynchrone avec Semantic Kernel

Cette cellule lance le processus d'analyse multi-agents. L'orchestration fonctionne en **mode asynchrone** pour gérer efficacement les appels API LLM.

**Composants techniques**:

| Composant | Rôle |
|-----------|------|
| `nest_asyncio.apply()` | Permet `await` dans Jupyter (normalement synchrone) |
| `run_analysis_conversation(texte)` | Fonction d'orchestration principale |
| `local_state` | Objet `RhetoricalAnalysisState` retourné |

**Étapes d'exécution internes**:

1. **Initialisation**: Création kernel SK, chargement plugins
2. **Tour 1 - Analyse informelle**: Agent LLM identifie arguments/sophismes
3. **Tour 2 - Formalisation**: Agent PL traduit en logique, exécute requêtes
4. **Synthèse**: Génération conclusion basée sur analyse hybride
5. **Retour**: État final JSON avec tous les résultats

> **Note performance**: L'exécution prend 90-150s selon la complexité du texte et la latence des API OpenAI/Anthropic. Surveillez les logs pour suivre la progression.

In [6]:
import nest_asyncio
import asyncio

# Lancer seulement si on a obtenu un texte valide ET que les définitions sont chargées
if 'texte_pour_analyse' in locals() and texte_pour_analyse and 'run_analysis_conversation' in locals():
    print("\n🚀 Lancement de l'exécution asynchrone de l'analyse...")
    nest_asyncio.apply()
    try:
        # Passer le texte préparé
        local_state = await run_analysis_conversation(texte_pour_analyse)
        print("\n🏁 Analyse terminée.")
    except Exception as e_analysis:
        print(f"\n❌ Une erreur est survenue pendant l'exécution de l'analyse : {e_analysis}")
        import traceback
        traceback.print_exc()

elif 'texte_pour_analyse' not in locals() or not texte_pour_analyse:
    print("\n Analyse non lancée : aucun texte n'a été préparé ou une erreur est survenue avant.")
else: # Implique que run_analysis_conversation n'a pas été chargée
     print("\n Analyse non lancée : la fonction d'orchestration n'a pas pu être chargée.")

01:20:10 [INFO] [Orchestration.Run.5908] --- Debut Nouveau Run ---


01:20:10 [INFO] [Orchestration.Run.5908] 1. Creation instance etat locale...


01:20:10 [INFO] [Orchestration.Run.5908]    Instance etat locale creee (id: 1676413977664) avec texte (longueur: 2348).


01:20:10 [INFO] [Orchestration.Run.5908] 2. Creation instance StateManagerPlugin locale...


01:20:10 [INFO] [Orchestration.StateManager] StateManagerPlugin initialisé avec l'instance RhetoricalAnalysisState (id: 1676413977664).


...

01:31:16 [INFO] [Orchestration.Run.5908] Fin analyse. Duree totale: 666.15 sec.


01:31:16 [INFO] [Orchestration.Run.5908] Log requetes PL: 5 entrees


01:31:16 [INFO] [Orchestration.Run.5908] Etat final JVM: (JVM non active)


01:31:16 [INFO] [Orchestration.Run.5908] --- Fin Run_5908 ---



--- Tour 12 (ProjectManagerAgent / ASSISTANT) ---
  Content: Conclusion finale rédigée et enregistrée. Résumé rapide et options suivantes :

- Quoi de fait : tâches 1→4 exécutées selon la séquence. Arguments et sophismes extraits et enregistrés ; belief set propositional_bs_1 créé. Requêtes PL générées et journalisées mais exécution bloquée (erreur JVM / Tweety non chargé).

- Recommandations enregistrées : maintenir parc nucléaire pilotable pendant la transition, investir en stockage et réseau, décarboner chaînes d'approvisionnement, études coût‑bénéfice...

--- Conversation Terminee ---

--- Historique Detaille ---
[Role:USER (USER)]: Bonjour a tous. Le texte a analyser est :
'''
Le debat sur la transition energetique en France revele des enjeux majeurs pour notre avenir.

ARGUMENTS EN FAVEUR DES ENERGIES RENOUVELABLES:
Les energies renouvelables sont essentielles pour lutter contre le changement climatique. La France s'est engagee a reduire ses emissions de CO2 de 40% d'ici 2030, c

### Interprétation du processus d'analyse

**Étapes d'exécution observables**:

1. **Analyse informelle** (InformalAnalysisAgent):
   - Identification des arguments principaux
   - Détection des sophismes via taxonomie
   - Stockage dans `local_state.identified_arguments` et `identified_fallacies`

2. **Formalisation logique** (PropositionalLogicAgent):
   - Traduction des arguments en propositions PL
   - Création de Belief Sets Tweety
   - Exécution de requêtes logiques (cohérence, implication)

3. **Synthèse collaborative**:
   - Combinaison des résultats informels et formels
   - Génération de la conclusion finale
   - Stockage dans `local_state.final_conclusion`

**Durée typique**: 90-150 secondes (selon complexité du texte et latence API)

**Logs à surveiller**:

| Message | Signification |
|---------|---------------|
| `[InformalAnalysisAgent] Processing...` | Analyse rhétorique en cours |
| `[PropositionalLogicAgent] Creating belief set...` | Formalisation PL |
| `FUNC_ERROR: JPype not initialized` | Tweety non disponible (fallback mode) |
| `Final state:` | Affichage JSON de l'état final |

> **Note de performance**: En mode batch, l'exécution complète prend quelques minutes (durée machine-dépendante : latence API LLM et complexité du texte ; plage typique donnée ci-dessus) sur le texte d'exemple enrichi.

### Exercice 2 : Compter les sophismes dans un rapport

Après une analyse collaborative, le rapport contient une liste de sophismes detectes. Ecrivez une fonction pour les compter.

**Objectif** : Implementer `count_fallacies`.

**Indices** :
- **Étape 1** : Parser la chaîne JSON avec `json.loads`
- **Étape 2** : Acceder a la cle `fallacies` (peut etre imbriquee)
- **Étape 3** : Compter les éléments de la liste
- **Étape 4** : Retourner 0 si la cle est absente

In [7]:
# Exercice 2 : Compter les sophismes dans un rapport JSON
# TODO etudiant: ecrivez une fonction qui extrait le nombre de sophismes
# d'un rapport d'analyse.
# Etape 1: Parser le JSON
# Etape 2: Acceder a la cle "fallacies"
# Etape 3: Compter les elements
# Etape 4: Gerer le cas ou la cle est absente
import json
def count_fallacies(report_str):
    return 0  # TODO etudiant

report = '{"analysis": {"fallacies": ["Ad Hominem", "Straw Man"]}}'
print(f"Nombre de sophismes: {count_fallacies(report)}")
print("Exercice a completer")


Nombre de sophismes: 0
Exercice a completer


## 5bis. Rapport de Validation de l'Analyse

### Validation structurée avec métriques de confiance

Cette section génère un **rapport de validation cross-validé** qui vérifie la complétude et la cohérence de l'analyse rhétorique.

**Objectif pédagogique**: Démontrer l'importance de la validation automatisée dans les systèmes d'IA, en particulier pour:
- Détecter les analyses incomplètes
- Calculer un score de confiance quantitatif
- Identifier les composants manquants
- Fournir un rapport exploitable (JSON)

**Méthode de validation**:

La validation cross-check vérifie 6 critères indépendants:

| Critère | Validation | Poids |
|---------|------------|-------|
| `ARGUMENTS_IDENTIFIED` | `len(identified_arguments) > 0` | 1/6 |
| `FALLACIES_ANALYZED` | `len(identified_fallacies) > 0` | 1/6 |
| `BELIEF_SET_CREATED` | `len(belief_sets) > 0` | 1/6 |
| `QUERIES_EXECUTED` | `len(query_log) > 0` | 1/6 |
| `QUERIES_MEANINGFUL` | Au moins 1 résultat ACCEPTED/REJECTED | 1/6 |
| `CONCLUSION_GENERATED` | `final_conclusion is not None` | 1/6 |

**Score de confiance**: `confidence = nombre_critères_validés / 6`

> **Application pratique**: Ce type de validation est essentiel dans les systèmes de production pour garantir la qualité des analyses automatisées. Le rapport JSON peut alimenter des tableaux de bord de monitoring.

In [8]:
# === CELLULE DE VALIDATION FINALE ===
# Genere un rapport JSON structure avec cross-validation

import json
from datetime import datetime
from typing import Dict, Any, List, Optional

def generate_validated_analysis_report(state) -> Dict[str, Any]:
    """Genere rapport JSON structure avec cross-validation."""
    
    report = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "version": "2.0-validated",
            "text_length": len(state.raw_text) if state.raw_text else 0,
            "text_snippet": (state.raw_text[:150] + "...") if state.raw_text and len(state.raw_text) > 150 else (state.raw_text or "")
        },
        "informal_analysis": {
            "arguments": [],
            "fallacies": [],
            "taxonomy_families_used": set()
        },
        "formal_analysis": {
            "belief_sets": [],
            "query_results": [],
            "consistency_checked": False
        },
        "cross_validation": {
            "validation_status": "INCOMPLETE",
            "confidence_score": 0.0,
            "checks_passed": [],
            "issues": []
        },
        "conclusion": {
            "summary": state.final_conclusion if hasattr(state, 'final_conclusion') else None,
            "is_complete": hasattr(state, 'final_conclusion') and state.final_conclusion is not None
        }
    }
    
    # Populate arguments
    if hasattr(state, 'identified_arguments'):
        for arg_id, arg_desc in state.identified_arguments.items():
            has_fallacy = False
            if hasattr(state, 'identified_fallacies'):
                has_fallacy = any(
                    f.get('target_argument_id') == arg_id
                    for f in state.identified_fallacies.values()
                )
            report["informal_analysis"]["arguments"].append({
                "id": arg_id, "description": str(arg_desc)[:200], "has_fallacy": has_fallacy
            })
    
    # Populate fallacies
    if hasattr(state, 'identified_fallacies'):
        for f_id, f_data in state.identified_fallacies.items():
            fallacy_type = f_data.get('type', 'Unknown') if isinstance(f_data, dict) else str(f_data)
            report["informal_analysis"]["fallacies"].append({
                "id": f_id,
                "type": fallacy_type,
                "justification": f_data.get('justification', '') if isinstance(f_data, dict) else '',
                "target_id": f_data.get('target_argument_id') if isinstance(f_data, dict) else None,
                "severity": "HIGH" if any(kw in str(fallacy_type).lower() for kw in ['manipulation', 'tromperie']) else "MEDIUM"
            })
            # Track taxonomy families
            if isinstance(fallacy_type, str) and '/' in fallacy_type:
                report["informal_analysis"]["taxonomy_families_used"].add(fallacy_type.split('/')[0])
    
    report["informal_analysis"]["taxonomy_families_used"] = list(report["informal_analysis"]["taxonomy_families_used"])
    
    # Populate belief sets
    if hasattr(state, 'belief_sets'):
        for bs_id, bs_data in state.belief_sets.items():
            content = bs_data.get('content', '') if isinstance(bs_data, dict) else str(bs_data)
            report["formal_analysis"]["belief_sets"].append({
                "id": bs_id,
                "logic_type": bs_data.get('logic_type', 'PL') if isinstance(bs_data, dict) else 'PL',
                "formula_count": content.count('\n') + 1 if content else 0,
                "is_consistent": "NOT_CHECKED"
            })
    
    # Populate query results
    if hasattr(state, 'query_log'):
        for qlog in state.query_log:
            raw_result = qlog.get('raw_result', '') if isinstance(qlog, dict) else ''
            status = "UNKNOWN"
            if "ACCEPTED" in str(raw_result): status = "ACCEPTED"
            elif "REJECTED" in str(raw_result): status = "REJECTED"
            elif "FUNC_ERROR" in str(raw_result): status = "ERROR"
            
            report["formal_analysis"]["query_results"].append({
                "log_id": qlog.get('log_id', '') if isinstance(qlog, dict) else '',
                "belief_set_id": qlog.get('belief_set_id', '') if isinstance(qlog, dict) else '',
                "query": qlog.get('query', '') if isinstance(qlog, dict) else '',
                "status": status
            })
    
    # === CROSS-VALIDATION LOGIC ===
    checks = []
    issues = []
    
    # Check 1: Arguments identified
    if len(report["informal_analysis"]["arguments"]) > 0:
        checks.append("ARGUMENTS_IDENTIFIED")
    else:
        issues.append("Aucun argument identifie")
    
    # Check 2: Fallacy analysis attempted
    if len(report["informal_analysis"]["fallacies"]) > 0:
        checks.append("FALLACIES_ANALYZED")
    elif hasattr(state, 'answers') and any("sophisme" in str(v).lower() for v in state.answers.values()):
        checks.append("FALLACY_ANALYSIS_ATTEMPTED")
    else:
        issues.append("Analyse sophismes non effectuee")
    
    # Check 3: Formal logic translation
    if len(report["formal_analysis"]["belief_sets"]) > 0:
        checks.append("BELIEF_SET_CREATED")
    else:
        issues.append("Aucun Belief Set PL cree")
    
    # Check 4: Queries executed
    if len(report["formal_analysis"]["query_results"]) > 0:
        checks.append("QUERIES_EXECUTED")
        accepted = sum(1 for q in report["formal_analysis"]["query_results"] if q["status"] == "ACCEPTED")
        rejected = sum(1 for q in report["formal_analysis"]["query_results"] if q["status"] == "REJECTED")
        if accepted > 0 or rejected > 0:
            checks.append("QUERIES_MEANINGFUL")
    else:
        issues.append("Aucune requete PL executee")
    
    # Check 5: Conclusion generated
    if report["conclusion"]["is_complete"]:
        checks.append("CONCLUSION_GENERATED")
    else:
        issues.append("Conclusion finale non generee")
    
    # Calculate confidence score
    max_checks = 6  # ARGUMENTS, FALLACIES, BELIEF_SET, QUERIES, QUERIES_MEANINGFUL, CONCLUSION
    confidence = len(checks) / max_checks
    report["cross_validation"]["confidence_score"] = round(confidence, 2)
    report["cross_validation"]["checks_passed"] = checks
    report["cross_validation"]["issues"] = issues
    
    # Determine validation status
    if confidence >= 0.8:
        report["cross_validation"]["validation_status"] = "COMPLETE_VALIDATED"
    elif confidence >= 0.5:
        report["cross_validation"]["validation_status"] = "PARTIAL_VALIDATED"
    elif confidence >= 0.3:
        report["cross_validation"]["validation_status"] = "MINIMAL"
    else:
        report["cross_validation"]["validation_status"] = "INCOMPLETE"
    
    return report


def display_validation_summary(report: Dict[str, Any]) -> str:
    """Affiche resume lisible du rapport de validation."""
    print("\n" + "="*70)
    print("          RAPPORT D'ANALYSE RHETORIQUE VALIDEE")
    print("="*70)
    
    cv = report["cross_validation"]
    status_symbols = {
        "COMPLETE_VALIDATED": "[OK]",
        "PARTIAL_VALIDATED": "[PARTIEL]",
        "MINIMAL": "[MINIMAL]",
        "INCOMPLETE": "[INCOMPLET]"
    }
    
    print(f"\n  STATUT: {status_symbols.get(cv['validation_status'], '?')} {cv['validation_status']}")
    print(f"  CONFIANCE: {cv['confidence_score']*100:.0f}%")
    
    print(f"\n  [ANALYSE INFORMELLE]")
    print(f"    Arguments: {len(report['informal_analysis']['arguments'])}")
    print(f"    Sophismes: {len(report['informal_analysis']['fallacies'])}")
    
    print(f"\n  [ANALYSE FORMELLE]")
    print(f"    Belief Sets: {len(report['formal_analysis']['belief_sets'])}")
    print(f"    Requetes: {len(report['formal_analysis']['query_results'])}")
    
    print(f"\n  [VALIDATIONS PASSEES]")
    for check in cv['checks_passed']:
        print(f"    [+] {check}")
    
    if cv['issues']:
        print(f"\n  [PROBLEMES DETECTES]")
        for issue in cv['issues']:
            print(f"    [-] {issue}")
    
    print(f"\n  [CONCLUSION]")
    if report['conclusion']['is_complete']:
        conclusion_preview = str(report['conclusion']['summary'])[:200]
        print(f"    {conclusion_preview}...")
    else:
        print("    (Non generee)")
    
    print("\n" + "="*70)
    
    return cv['validation_status']


# === EXECUTION DE LA VALIDATION ===
print("\n--- Generation du Rapport d'Analyse Validee ---")

# Chercher l'etat local_state defini par l'orchestration
if 'local_state' in dir() and local_state is not None:
    # Generate report
    final_report = generate_validated_analysis_report(local_state)
    
    # Display summary
    validation_status = display_validation_summary(final_report)
    
    # Export JSON
    output_path = "output/analysis_report.json"
    try:
        import os
        os.makedirs("output", exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_report, f, indent=2, ensure_ascii=False, default=str)
        print(f"\nRapport JSON exporte vers: {output_path}")
    except Exception as e:
        print(f"Erreur export JSON: {e}")
    
    # Afficher JSON complet
    print("\n--- RAPPORT JSON COMPLET ---")
    print(json.dumps(final_report, indent=2, ensure_ascii=False, default=str))
    
    # Final verdict
    print("\n--- VERDICT FINAL ---")
    if validation_status == "COMPLETE_VALIDATED":
        print("[SUCCESS] ANALYSE RHETORIQUE VALIDEE COMPLETE")
        print("  Tous les criteres de validation sont satisfaits.")
    elif validation_status == "PARTIAL_VALIDATED":
        print("[PARTIEL] Analyse partiellement validee")
        print("  Certaines etapes manquent pour une validation complete.")
    else:
        print(f"[{validation_status}] Analyse incomplete")
        print("  Verifiez les problemes detectes ci-dessus.")

else:
    print("[INFO] Etat d'analyse (local_state) non disponible.")
    print("  Executez d'abord les cellules precedentes pour lancer l'analyse.")
    print("  Ou l'analyse n'a pas ete executee (mode batch sans erreur?)")


--- Generation du Rapport d'Analyse Validee ---

          RAPPORT D'ANALYSE RHETORIQUE VALIDEE

  STATUT: [OK] COMPLETE_VALIDATED
  CONFIANCE: 83%

  [ANALYSE INFORMELLE]
    Arguments: 4
    Sophismes: 3

  [ANALYSE FORMELLE]
    Belief Sets: 1
    Requetes: 5

  [VALIDATIONS PASSEES]
    [+] ARGUMENTS_IDENTIFIED
    [+] FALLACIES_ANALYZED
    [+] BELIEF_SET_CREATED
    [+] QUERIES_EXECUTED
    [+] CONCLUSION_GENERATED

  [CONCLUSION]
    Synthèse finale enregistrée :
- Résumé des arguments pour les renouvelables (climat, emplois, exemples internationaux) et contre (coût, intermittence, rôle du nucléaire).
- Trois sophismes principaux ...


Rapport JSON exporte vers: output/analysis_report.json

--- RAPPORT JSON COMPLET ---
{
  "metadata": {
    "timestamp": "2026-09-25T01:31:16.326053",
    "version": "2.0-validated",
    "text_length": 2348,
    "text_snippet": "Le debat sur la transition energetique en France revele des enjeux majeurs pour notre avenir.\n\nARGUMENTS EN FAVEUR DES

### Interprétation du rapport de validation

**Statuts de validation possibles**:

| Statut | Score | Critères |
|--------|-------|----------|
| `COMPLETE_VALIDATED` | ≥80% | Tous les composants critiques présents |
| `PARTIAL_VALIDATED` | ≥50% | Analyse fonctionnelle mais incomplète |
| `MINIMAL` | ≥30% | Quelques éléments détectés |
| `INCOMPLETE` | <30% | Échec de l'analyse |

**Critères de validation**:

1. **ARGUMENTS_IDENTIFIED**: Au moins un argument détecté
2. **FALLACIES_ANALYZED**: Au moins un sophisme identifié
3. **BELIEF_SET_CREATED**: Au moins un Belief Set PL généré
4. **QUERIES_EXECUTED**: Au moins une requête logique lancée
5. **QUERIES_MEANINGFUL**: Résultat ACCEPTED ou REJECTED obtenu
6. **CONCLUSION_GENERATED**: Conclusion finale présente

**Exemple d'analyse réussie**:

```json
{
  "cross_validation": {
    "validation_status": "COMPLETE_VALIDATED",
    "confidence_score": 0.83,
    "checks_passed": [
      "ARGUMENTS_IDENTIFIED",
      "FALLACIES_ANALYZED",
      "BELIEF_SET_CREATED",
      "QUERIES_EXECUTED",
      "CONCLUSION_GENERATED"
    ],
    "issues": []
  }
}
```

> **Utilité du rapport JSON**: Le fichier `output/analysis_report.json` peut être utilisé pour:
> - Tests automatisés (CI/CD)
> - Métriques de qualité d'analyse
> - Comparaison entre versions du système
> - Documentation des résultats d'analyse

### Exercice 3 : Mapper un score de confiance a un statut

Le rapport de validation attribue des scores de confiance aux analyses. Convertissez ces scores en labels comprehensibles.

**Objectif** : Implementer `confidence_to_status`.

**Indices** :
- **Étape 1** : Définir les seuils : >0.8 = VALIDATED, 0.5-0.8 = UNCERTAIN, <0.5 = LOW
- **Étape 2** : Comparer le score aux seuils avec des `if/elif/else`
- **Étape 3** : Retourner le label correspondant
- **Étape 4** : Gerer les cas extremes (score < 0 ou > 1)

In [9]:
# Exercice 3 : Mapper un score de confiance a un statut
# TODO etudiant: ecrivez une fonction qui classifie un score de confiance
# en categories: VALIDATED (>0.8), UNCERTAIN (0.5-0.8), LOW (<0.5).
# Etape 1: Definir les seuils
# Etape 2: Comparer le score aux seuils
# Etape 3: Retourner le label correspondant
# Etape 4: Gerer les valeurs hors plage
def confidence_to_status(score):
    return "UNKNOWN"  # TODO etudiant

for s in [0.95, 0.65, 0.3]:
    print(f"Score {s}: {confidence_to_status(s)}")
print("Exercice a completer")


Score 0.95: UNKNOWN
Score 0.65: UNKNOWN
Score 0.3: UNKNOWN
Exercice a completer


## 6. Résultats et Conclusion

Vérifiez les logs et l'état final JSON affichés par l'exécution précédente pour voir le résultat de l'analyse collaborative.

### Récapitulatif du workflow complet

Le notebook orchestrateur `Argumentation-08b-Executor-Python.ipynb` implémente un pipeline d'analyse rhétorique en 7 étapes:

| Étape | Action | Durée typique |
|-------|--------|---------------|
| 1. Configuration | Chargement `.env`, variables d'environnement | <1s |
| 2. Interface/Batch | Configuration texte (UI ou batch) | Variable (0-60s) |
| 3. Chargement agents | Import des 4 notebooks modulaires | 5-10s |
| 4. Analyse informelle | Détection arguments + sophismes | 40-60s |
| 5. Formalisation PL | Création Belief Sets + requêtes | 30-50s |
| 6. Validation | Génération rapport JSON structuré | <1s |
| 7. Export | Sauvegarde `output/analysis_report.json` | <1s |

**Total (mode batch)**: ~120-150 secondes

### Méthodologie d'analyse rhétorique

Le système combine deux approches complémentaires:

**Analyse informelle (InformalAnalysisAgent)**:
- Utilise un LLM (GPT-4/Claude) pour comprendre le langage naturel
- Identifie arguments, prémisses, conclusions
- Détecte sophismes via taxonomie hiérarchique
- Output: Structure JSON décrivant la rhétorique

**Analyse formelle (PropositionalLogicAgent)**:
- Traduit les arguments en logique propositionnelle
- Crée des Belief Sets Tweety (bibliothèque Java)
- Exécute requêtes de cohérence et d'implication
- Output: Preuves formelles ou contre-exemples

> **Avantage de l'approche hybride**: La combinaison LLM + logique formelle permet à la fois la compréhension contextuelle (LLM) et la rigueur mathématique (logique).

## 7. Pistes d'Amélioration Futures

### Améliorations techniques

| Catégorie | Amélioration proposée | Impact |
|-----------|----------------------|--------|
| **Logique formelle** | Implémenter FOL, logique modale (Tweety) | Expressivité accrue |
| **Taxonomie sophismes** | Navigation profondeur variable, exemples | Précision détection |
| **Gestion erreurs** | Retry automatique, fallback gracieux | Robustesse production |
| **Performance** | Cache LLM, parallélisation agents | Réduction latence |
| **État RDF/KG** | Migration vers graphe sémantique (rdflib) | Requêtes complexes |

### Améliorations pédagogiques

**Visualisations interactives**:
- Graphe d'arguments (NetworkX + Plotly)
- Arbre taxonomique des sophismes
- Timeline de l'analyse (Gantt chart)

**Explications enrichies**:
- Définitions des sophismes avec exemples historiques
- Comparaison analyse informelle vs formelle
- Métriques de confiance par argument

**Interface utilisateur**:
- Dashboard Gradio/Streamlit pour upload de textes
- Annotation collaborative des arguments
- Export PDF avec rapport structuré

### Cas d'usage avancés

**Analyse de débats politiques**:
- Détection de manipulation rhétorique
- Comparaison positions de candidats
- Fact-checking automatisé (intégration sources externes)

**Éducation à la pensée critique**:
- Exercices interactifs sur textes annotés
- Quiz génératif sur détection de sophismes
- Feedback personnalisé sur essais d'étudiants

**Recherche académique**:
- Corpus analysis (analyse de milliers de textes)
- Évolution diachronique de l'argumentation
- Comparaison cultures/langues (multilingual)

> **Roadmap prioritaire**: 
> 1. Finaliser intégration Tweety (Query exécution)
> 2. Implémenter visualisations graphiques
> 3. Ajouter interface Gradio
> 4. Publier package Python réutilisable